# Cotton benchmark — teacher training (YOLO11s)
Trains the annotation teacher on the Roboflow set, excluding all tiles from
test/buffer regions per `roboflow_audit.csv`.

**Drive `cotton_bench/` must contain:** the Roboflow zip
(`cotton-ball-deetction.v1i.yolov11.zip`) and `roboflow_audit.csv`.
Runtime: T4 GPU. Training ~1–2 h; safe to leave running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
import os, glob
zips = glob.glob(f'{DRIVE}/*.v1i.yolov11*.zip') + glob.glob(f'{DRIVE}/*roboflow*.zip')
assert zips, 'Roboflow zip not found in Drive/cotton_bench'
assert os.path.exists(f'{DRIVE}/roboflow_audit.csv'), 'roboflow_audit.csv missing'
RF_ZIP = zips[0]; print('using', RF_ZIP)

In [ ]:
!pip install -q ultralytics
!unzip -q "{RF_ZIP}" -d /content/rf_raw
import glob
print('train imgs:', len(glob.glob('/content/rf_raw/train/images/*')))
print('valid imgs:', len(glob.glob('/content/rf_raw/valid/images/*')))
print('test  imgs:', len(glob.glob('/content/rf_raw/test/images/*')))

In [ ]:
# Build filtered dataset: drop every tile whose source is our_TEST/our_BUFFER
import csv, os, shutil

exclude = set()
with open(f'{DRIVE}/roboflow_audit.csv') as f:
    for row in csv.DictReader(f):
        if row['location'] in ('our_TEST', 'our_BUFFER'):
            exclude.add(row['file'])
print('excluding', len(exclude), 'tiles')

kept = dropped = 0
for split in ('train', 'valid', 'test'):
    for sub in ('images', 'labels'):
        os.makedirs(f'/content/rf_clean/{split}/{sub}', exist_ok=True)
    for img in glob.glob(f'/content/rf_raw/{split}/images/*'):
        name = os.path.basename(img)
        if name in exclude:
            dropped += 1
            continue
        lbl = img.replace('/images/', '/labels/').rsplit('.',1)[0] + '.txt'
        shutil.copy(img, f'/content/rf_clean/{split}/images/{name}')
        if os.path.exists(lbl):
            shutil.copy(lbl, f'/content/rf_clean/{split}/labels/')
        kept += 1
print(f'kept {kept}, dropped {dropped}')

with open('/content/rf_clean/data.yaml', 'w') as f:
    f.write('path: /content/rf_clean\ntrain: train/images\n'
            'val: valid/images\ntest: test/images\n'
            'names:\n  0: cotton_boll\n')

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
results = model.train(
    data='/content/rf_clean/data.yaml',
    epochs=150, patience=30, imgsz=640, batch=16,
    project='/content/runs', name='teacher_11s',
    max_det=3000, seed=42)

In [ ]:
# Validate and save teacher to Drive
import shutil
best = '/content/runs/teacher_11s/weights/best.pt'
m = YOLO(best)
metrics = m.val(data='/content/rf_clean/data.yaml', split='val', max_det=3000)
print('val mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)
shutil.copy(best, f'{DRIVE}/teacher_11s_best.pt')
print('teacher saved to Drive: teacher_11s_best.pt')